In [27]:
import xarray as xr
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd

import seaborn as sns
sns.set_style("whitegrid")
from matplotlib import colors

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [28]:
base = "outputs"

In [29]:
pwd

'/Users/sasthana/curnagl_shivanshi/Downscaling/Processing_and_Analysis_Scripts/Analysis/Paper_Stats'

In [30]:
file=pd.read_csv("/Users/sasthana/curnagl_shivanshi/Downscaling/Processing_and_Analysis_Scripts/Analysis/Paper_Stats/SR_metrics_cobweb.csv")

In [17]:
file

,Unnamed: 0,CRPS_temp,LSD_temp_ensmedian,SSIM_temp_ensmedian,RMSE_temp_ensmedian,MAE_temp_ensmedian,PSNR_temp_ensmedian,PITD_temp,CRPS_precip,LSD_precip_ensmedian,SSIM_precip_ensmedian,RMSE_precip_ensmedian,MAE_precip_ensmedian,PSNR_precip_ensmedian,PITD_precip
0,Coarse_temp,1.363373,0.276881,0.704019,1.464239,0.062357,49.413392,0.196847,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bicubic_temp,1.296814,0.267778,0.731952,1.392674,0.030949,56.894729,0.197194,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Bilinear_temp,1.347273,0.276520,0.724461,1.448104,0.026815,58.107019,0.197015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,UNet_temp,0.258314,0.194768,0.976858,0.336881,0.070162,50.113015,0.164231,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DDIM_temp,0.192403,0.193895,0.976556,0.330701,0.061101,51.295399,0.080787,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Coarse_precip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.554616,0.234346,0.760182,1.277110,0.226908,43.082026,0.129154
6,Bicubic_precip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.395723,0.167032,0.792797,0.890246,0.267172,41.684295,0.129368
7,Bilinear_precip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.442332,0.192596,0.800487,1.024051,0.261773,41.868318,0.128873
8,UNet_precip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.309757,0.142925,0.710349,0.696390,0.275051,41.160478,0.156580
9,DDIM_precip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.254852,0.139345,0.824096,0.709553,0.255016,41.204766,0.103744


In [ ]:
models = ["Coarse", "Bicubic", "UNet", "DDIM(10 samples)"]
metrics = ["CRPS ↓", "LSD ↓", "PITD ↓", "RMSE ↓", "1-SSIM↓ "]

In [5]:
def get_metric(file, col):
    df = pd.read_csv(file)
    if col not in df.columns:
        col = df.columns[1]
    return df.set_index(df.columns[0])[col].reindex(models).values


In [6]:
metric_keys = ["CRPS", "LSD", "PITD", "RMSE", "SSIM"]
metrics = ["CRPS ↓", "LSD ↓", "PITD ↓", "RMSE ↓", "1-SSIM ↓"]

def collect_data(var):
    data = []
    for metric in metric_keys:
        if metric == "SSIM":
            vals = get_metric(files[metric][var], "SSIM")
            vals = 1 - vals  # Calculate 1-SSIM
        elif metric == "RMSE":
            vals = get_metric(files[metric][var], "RMSE")
        else:
            vals = get_metric(files[metric][var], metric)
        data.append(vals)
    return np.array(data).T

In [7]:
def get_best_ddim(metric_file, best_col):
    df = pd.read_csv(metric_file)
    return df.loc[df.iloc[:,0] == "DDIM(11 samples)", best_col].values[0]

In [8]:
def inject_best(data, var):
    data = data.copy()
    # For temp
    if var == "temp":
        best_ssim = get_best_ddim(files["SSIM"]["temp"], "Best SSIM")
        best_rmse = get_best_ddim(files["RMSE"]["temp"], "Best RMSE")
        if not pd.isna(best_ssim):
            data[3, 4] = 1 - best_ssim  # Convert SSIM to 1-SSIM
        if not pd.isna(best_rmse):
            data[3, 3] = best_rmse  # RMSE is 4th col
    # For precip
    if var == "precip":
        best_ssim = get_best_ddim(files["SSIM"]["precip"], "Best SSIM")
        best_rmse = get_best_ddim(files["RMSE"]["precip"], "Best RMSE")
        if not pd.isna(best_ssim):
            data[3, 4] = 1 - best_ssim  # Convert SSIM to 1-SSIM
        if not pd.isna(best_rmse):
            data[3, 3] = best_rmse
    return data


In [9]:
def plot_radar(data, title, metrics, models):


    N = len(metrics)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # close the loop

    # Use tab10 for colorblind-friendly palette
    color_names = list(mcolors.TABLEAU_COLORS)
    colors = [mcolors.TABLEAU_COLORS[name] for name in color_names][:len(models)]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    for i, model in enumerate(models):
        values = data[i].tolist()
        values += values[:1]
        ax.plot(angles, values, label=model, linewidth=3, color=colors[i])
        ax.fill(angles, values, alpha=0.15, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=16, fontweight='bold')

    ax.tick_params(axis='y', labelsize=18)
    ax.grid(True, linestyle='--', linewidth=1, alpha=0.7)

    ax.set_title(title, size=30, y=1.12, fontweight='bold')
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=14, frameon=True)

    plt.figtext(0.5, 0.01, "↓: Lower is better", ha='center', fontsize=14, fontstyle='italic')
    plt.savefig(f"{title.replace(' ', '_')}.png", dpi=1000, bbox_inches='tight')
    plt.show()

In [10]:

metric_keys = ["CRPS", "LSD", "PITD", "RMSE", "1-SSIM"]
metrics = ["CRPS ↓", "LSD ↓", "PITD ↓", "RMSE ↓", "1-SSIM↓ "]

def collect_data(var):
    data = []
    for metric in metric_keys:
        if metric == "1-SSIM":
            vals = get_metric(files["SSIM"][var], "SSIM")
            vals = 1 - vals  # Calculate 1-SSIM
        elif metric == "RMSE":
            vals = get_metric(files[metric][var], "RMSE")
        else:
            vals = get_metric(files[metric][var], metric)
        data.append(vals)
    return np.array(data).T



In [11]:

data_temp = collect_data("temp")
data_temp = inject_best(data_temp, "temp")

data_precip = collect_data("precip")
data_precip = inject_best(data_precip, "precip")

plot_radar(data_temp, "Daily Temperature SR metrics for test set (2011-2023)", metrics, models)
plot_radar(data_precip, "Daily Precipitation SR metrics for test set (2011-2023)", metrics, models)


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/crps_allmodels_temp.csv'

Visualising single observational run frame 

Paths